# Two-stage recommender on MovieLens - Colab notebook

Runs the whole pipeline: Spark features, two-tower retrieval (TF Recommenders), FAISS index, DCN and Wide & Deep rankers, evaluation, a simulated A/B test, and a small online demo with Redis and cold-start fallbacks.

Before you start: Runtime > Change runtime type > T4 GPU. Run the cells in order. All numbers shown come from your own run.

`MODE = "small"` uses ml-latest-small (about 100k ratings) and finishes quickly, so you can check that everything works. `MODE = "full"` uses ml-25m with Spark and the settings in `configs/config.yaml`.

In [ ]:
MODE = "small"   # Keep "small" for the first successful run. Switch to "full" only later.
assert MODE in ("small", "full"), "MODE must be small or full"
SETTINGS = {
    "small": ["data.dataset=ml-latest-small", "data.engine=pandas", "retrieval.batch_size=512",
              "retrieval.epochs=15", "retrieval.n_candidates=100", "ranker.epochs=4",
              "eval.max_eval_users=600"],
    "full": [],
}[MODE]
OVERRIDES = [arg for setting in SETTINGS for arg in ("--set", setting)]
print("Running mode:", MODE, "| overrides:", SETTINGS or "project defaults")

## 1. Get the code
Upload `recsys-engine.zip` through the file panel on the left, or put your GitHub repo URL in `GIT_URL`.

In [ ]:
import os, sys, glob, shutil, zipfile, subprocess
from pathlib import Path

GIT_URL = ""  # Leave empty when uploading recsys-engine.zip.
workdir = Path("/content")
os.chdir(workdir)
project = workdir / "recsys-engine"
if project.exists():
    # This allows re-running the entire notebook with a clean copy of the uploaded project.
    # Download the results_bundle.zip before a new run; generated artifacts inside project are removed.
    shutil.rmtree(project)
if GIT_URL:
    subprocess.run(["git", "clone", GIT_URL, str(project)], check=True)
else:
    uploads = sorted(workdir.glob("recsys-engine*.zip"))
    assert uploads, "Upload recsys-engine.zip in the Colab Files panel first"
    with zipfile.ZipFile(uploads[0]) as archive:
        # Reject zip paths that could escape /content.
        for member in archive.namelist():
            path = Path(member)
            assert not path.is_absolute() and ".." not in path.parts, f"Unsafe ZIP path: {member}"
        archive.extractall(workdir)
assert (project / "configs/config.yaml").is_file(), "Project could not be extracted"
os.chdir(project)
os.environ["RECSYS_HOME"] = str(project)
os.environ["PYTHONPATH"] = str(project / "src")
os.environ["TF_USE_LEGACY_KERAS"] = "1"  # Must be set before TensorFlow loads.
sys.path.insert(0, str(project / "src"))
print("Project loaded:", project)

def run_stage(stage):
    print(f"\n===== {stage} ({MODE}) =====", flush=True)
    subprocess.run([sys.executable, "-m", "recsys.cli", stage, *OVERRIDES], check=True)

# A failed command now raises an error and stops the notebook instead of being silently ignored.
def run_check(args):
    print("\n=====", " ".join(args), "=====", flush=True)
    subprocess.run([sys.executable, *args], check=True)

## 2. Install packages
The notebook checks your existing Colab TensorFlow version and installs the matching legacy `tf-keras` package before using TensorFlow Recommenders. It also installs the API demo dependencies. Any failed installation stops the run.

In [ ]:
import importlib.metadata
from pathlib import Path

# Read the installed version without importing TensorFlow before tf-keras is installed.
tf_version = importlib.metadata.version("tensorflow")
major, minor = tf_version.split(".")[:2]
packages = [f"tf-keras~={major}.{minor}.0", "tensorflow-recommenders>=0.7.3",
            "faiss-cpu>=1.8", "pyspark>=3.5,<4", "redis>=5.0", "pyarrow>=14",
            "pyyaml>=6", "tritonclient[http]>=2.40", "matplotlib>=3.8",
            "fastapi>=0.110", "uvicorn>=0.29", "httpx>=0.27"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)
subprocess.run(["apt-get", "-qq", "install", "-y", "redis-server"], check=True)
subprocess.run(["redis-server", "--daemonize", "yes"], check=True)
subprocess.run(["redis-cli", "ping"], check=True)
subprocess.run(["java", "-version"], check=True)
import tensorflow as tf
print("TensorFlow:", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))
assert tf.__version__.split(".")[:2] == [major, minor], "TensorFlow changed unexpectedly"

## 3. Checks
First run the 19 original unit tests plus new regression checks. Then execute the synthetic end-to-end pipeline using Pandas and Spark. **The notebook stops on the first failure.** Do not interpret synthetic smoke-test numbers as real model results.

In [ ]:
run_check(["-m", "unittest", "discover", "-s", "tests", "-v"])
run_check(["scripts/smoke_test.py"])

In [ ]:
run_check(["scripts/smoke_test.py", "--engine", "spark"])

## 4. Data: download, clean, split by time, build features

In [ ]:
for stage in ("download", "split", "features"):
    run_stage(stage)

In [ ]:
import pandas as pd
for name in ["train", "val", "test"]:
    t = pd.read_parquet(f"data/processed/{name}.parquet")
    print(f"{name:5s} {len(t):>10,} rows   timestamps {t.timestamp.min()} .. {t.timestamp.max()}")

## 5. Stage 1: two-tower retrieval and the FAISS index

In [ ]:
for stage in ("train_retrieval", "build_index"):
    run_stage(stage)

## 6. Stage 2: rankers (DCN is arm A, Wide & Deep is arm B)

In [ ]:
run_stage("train_ranker")

## 7. Offline evaluation on the test period
Compares the popularity baseline, retrieval alone and retrieval plus each ranker, and also reports how closely the approximate index matches exact search and how cold-start users fare.

In [ ]:
run_stage("evaluate")

## 8. A/B test and report
The clicks here are simulated from held-out ratings (see the README), so treat this as a check of the machinery, not as evidence about real users.

In [ ]:
run_stage("ab_replay")
run_check(["scripts/make_report.py"])
print((project / "docs/RESULTS.md").read_text())

In [ ]:
from IPython.display import Image
Image("reports/ndcg.png")

## 9. Online demo: real `/recommend` API route, Redis and cold start
FastAPI's TestClient calls the actual API endpoints in-process, so Colab does not need a public tunnel or Docker. This is a working endpoint demonstration, not proof that Triton is running. Keep the output visible for a screen recording.

In [ ]:
from fastapi.testclient import TestClient
from recsys.serving.api import app
from recsys.pipeline.stages import build_recommender
from recsys.config import load_config

# The API loads configs/config.yaml by default. For the small mode, temporarily
# point it at an override config so it uses the models just trained in Colab.
import yaml
cfg = load_config(overrides=SETTINGS)
config_file = project / "configs/colab_active.yaml"
config_file.write_text(yaml.safe_dump(cfg))
os.environ["RECSYS_CONFIG"] = str(config_file)

with TestClient(app) as client:
    health = client.get("/health")
    health.raise_for_status()
    print("GET /health:", health.json())
    known_user = int(build_recommender(cfg).fs.user_ids[0])
    payload = {"user_id": known_user, "k": 5}
    response = client.post("/recommend", json=payload)
    response.raise_for_status()
    result = response.json()
    print("POST /recommend (first):", result)
    cached = client.post("/recommend", json=payload)
    cached.raise_for_status()
    print("POST /recommend (second):", cached.json())
    assert cached.json()["source"] == "cache", "Caching failed"
    cold = client.post("/recommend", json={"user_id": 999999999, "k": 5,
                                          "liked_genres": ["Sci-Fi", "Thriller"]})
    cold.raise_for_status()
    print("POST /recommend (new user):", cold.json())
    assert cold.json()["source"] in ("cold_content", "cache"), "Cold-start recommendation failed"
    assert cold.json()["user_id"] == 999999999, "Cold-start cache returned another user ID"
print("API demo: SUCCESS. A/B replay results elsewhere in this notebook are simulated.")

## 10. Export model repository for Triton and download submission results
The export creates a Triton model repository, but **does not run Triton** (Docker/x86 Linux is required). The small `results_bundle.zip` includes the real run's reports, documentation, and Triton configurations, not heavy model weights. Keep `recsys-engine.zip` as your source project.

In [ ]:
import zipfile
run_stage("export_triton")
# Save lightweight results for merging into the GitHub project. Do NOT bundle
# ranker weights or the exported Triton models; those can be very large.
output_zip = Path("/content/results_bundle.zip")
with zipfile.ZipFile(output_zip, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as out:
    for directory in ("reports", "docs"):
        for file in (project / directory).rglob("*"):
            if file.is_file() and file.name != ".gitkeep":
                out.write(file, file.relative_to(project))
    repo = project / "triton/model_repository"
    for file in repo.rglob("*"):
        if file.is_file() and file.name in ("config.pbtxt", "EXPORT_INFO.json"):
            out.write(file, file.relative_to(project))
print("Results ZIP created:", output_zip, "size (MB):", round(output_zip.stat().st_size / 1e6, 2))
from google.colab import files
files.download(str(output_zip))